# Benign Flow Integration — CIC18 to UNSW15

This notebook evaluates the integration of benign network flows from **GenIDS-CIC18** into **GenIDS-UNSW15** for cross-dataset IDS generalization experiments.

The integration procedure follows three main principles:

1. A percentage of benign flows is selected from CIC18 and integrated into the UNSW15 training dataset.
2. The integrated CIC18 flows are removed from the CIC18 evaluation dataset to prevent data leakage.
3. The same percentage of benign flows is removed from UNSW15 before integration, preserving the dataset size and the class distribution as much as possible.

The default configuration uses a **20% benign flow integration rate**. The same notebook can be reused for 40%, 60%, and 80% by changing the `INTEGRATION_RATE` parameter.

## 1. Environment Setup

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

## 2. Experiment Configuration

Adjust the paths below according to your local environment or repository structure.

In [ ]:
RANDOM_STATE = 42
INTEGRATION_RATE = 0.20  # Change to 0.40, 0.60, or 0.80 for the remaining scenarios.
TEST_SIZE = 0.30

DATA_DIR = Path("../datasets")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NB15_PATH = DATA_DIR / "GenIDS-NB15.csv"
CIC18_PATH = DATA_DIR / "GenIDS-CIC18.csv"

LABEL_COLUMN = "binary"
TIME_COLUMN = "bidirectional_first_seen_ms"
SOURCE_COLUMN = "source_dataset"

print(f"Integration rate: {INTEGRATION_RATE:.0%}")

## 3. Helper Functions

In [ ]:
def load_dataset(path: Path, dataset_name: str) -> pd.DataFrame:
    """Load a dataset from a CSV file."""
    if not path.exists():
        raise FileNotFoundError(
            f"File not found for {dataset_name}: {path}. "
            "Update DATA_DIR or the file name in the configuration cell."
        )
    dataframe = pd.read_csv(path)
    print(f"{dataset_name}: {dataframe.shape}")
    return dataframe


def remove_non_model_columns(dataframe: pd.DataFrame, columns_to_remove: list[str]) -> pd.DataFrame:
    """Remove columns that should not be used during model training."""
    return dataframe.drop(columns=columns_to_remove, errors="ignore").copy()


def normalize_label_column(dataframe: pd.DataFrame, label_column: str = LABEL_COLUMN) -> pd.DataFrame:
    """Create a standardized binary label column when the dataset uses another label name."""
    dataframe = dataframe.copy()
    if label_column not in dataframe.columns and "label" in dataframe.columns:
        dataframe[label_column] = dataframe["label"]
    return dataframe


def encode_categorical_columns(dataframes: list[pd.DataFrame], categorical_columns: list[str]) -> list[pd.DataFrame]:
    """Encode categorical columns independently for each dataset."""
    encoded_dataframes = []
    for dataframe in dataframes:
        dataframe = dataframe.copy()
        for column in categorical_columns:
            if column in dataframe.columns:
                dataframe[column] = LabelEncoder().fit_transform(dataframe[column].astype(str))
        encoded_dataframes.append(dataframe)
    return encoded_dataframes


def standardize_numeric_types(dataframe: pd.DataFrame, label_column: str = LABEL_COLUMN) -> pd.DataFrame:
    """Convert numeric feature columns to float64 and the label column to int64."""
    dataframe = dataframe.copy()
    for column in dataframe.columns:
        if column == label_column:
            dataframe[column] = dataframe[column].astype(np.int64)
        elif pd.api.types.is_numeric_dtype(dataframe[column]):
            dataframe[column] = dataframe[column].astype(np.float64)
    return dataframe


def print_dataset_summary(dataframe: pd.DataFrame, dataset_name: str, label_column: str = LABEL_COLUMN) -> None:
    """Print dataset shape and class distribution."""
    print(f"\n{dataset_name}")
    print(f"Shape: {dataframe.shape}")
    print("Class distribution:")
    print(dataframe[label_column].value_counts().sort_index())
    print("Class distribution (%):")
    print(dataframe[label_column].value_counts(normalize=True).sort_index().map("{:.2%}".format))


def select_first_benign_flows(dataframe: pd.DataFrame, rate: float, label_column: str = LABEL_COLUMN) -> pd.DataFrame:
    """Select the first percentage of benign flows from a dataset."""
    benign_flows = dataframe[dataframe[label_column] == 0]
    number_of_flows = int(len(benign_flows) * rate)
    return benign_flows.iloc[:number_of_flows].copy()


def integrate_benign_flows(
    target_dataframe: pd.DataFrame,
    source_dataframe: pd.DataFrame,
    integration_rate: float,
    target_name: str = "UNSW15",
    source_name: str = "CIC18",
    label_column: str = LABEL_COLUMN,
    time_column: str = TIME_COLUMN,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Integrate benign flows from the source dataset into the target dataset.

    Returns:
        integrated_target: target dataset after removing benign flows and adding source benign flows.
        remaining_source: source dataset after removing the integrated flows.
        selected_source_benign: benign source flows integrated into the target dataset.
        removed_target_benign: benign target flows removed before integration.
    """
    selected_source_benign = select_first_benign_flows(source_dataframe, integration_rate, label_column)
    removed_target_benign = select_first_benign_flows(target_dataframe, integration_rate, label_column)

    remaining_source = source_dataframe.drop(index=selected_source_benign.index).copy()
    target_without_selected_benign = target_dataframe.drop(index=removed_target_benign.index).copy()

    target_without_selected_benign[SOURCE_COLUMN] = target_name.lower()
    selected_source_benign[SOURCE_COLUMN] = f"{source_name.lower()}_{int(integration_rate * 100)}_benign"

    integrated_target = pd.concat(
        [target_without_selected_benign, selected_source_benign],
        ignore_index=True,
    )

    if time_column in integrated_target.columns:
        integrated_target = integrated_target.sort_values(by=time_column).reset_index(drop=True)

    return integrated_target, remaining_source, selected_source_benign, removed_target_benign


def split_and_scale(
    train_dataframe: pd.DataFrame,
    external_test_dataframes: dict[str, pd.DataFrame],
    label_column: str = LABEL_COLUMN,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
):
    """Split the integrated target dataset and scale all datasets using the training split."""
    feature_dataframe = train_dataframe.drop(columns=[label_column, SOURCE_COLUMN], errors="ignore")
    label_series = train_dataframe[label_column]

    X_train, X_test, y_train, y_test = train_test_split(
        feature_dataframe,
        label_series,
        test_size=test_size,
        random_state=random_state,
        stratify=label_series,
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    scaled_external_sets = {}
    for dataset_name, dataframe in external_test_dataframes.items():
        X_external = dataframe.drop(columns=[label_column, SOURCE_COLUMN], errors="ignore")
        y_external = dataframe[label_column]
        scaled_external_sets[dataset_name] = (scaler.transform(X_external), y_external)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaled_external_sets, feature_dataframe.columns


def build_xgboost_model(random_state: int = RANDOM_STATE) -> XGBClassifier:
    """Create the XGBoost classifier used in the experiment."""
    return XGBClassifier(
        eval_metric="logloss",
        n_estimators=300,
        max_depth=10,
        objective="binary:logistic",
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.5,
        random_state=random_state,
        n_jobs=-1,
    )


def evaluate_binary_classifier(model, X, y, dataset_name: str) -> dict[str, float]:
    """Evaluate a binary classifier and return the main IDS metrics."""
    y_pred = model.predict(X)
    y_score = model.predict_proba(X)[:, 1]

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(y, y_pred),
        "Precision_macro": precision_score(y, y_pred, average="macro", zero_division=0),
        "Recall_macro": recall_score(y, y_pred, average="macro", zero_division=0),
        "F1_macro": f1_score(y, y_pred, average="macro", zero_division=0),
        "AUC_ROC": roc_auc_score(y, y_score),
        "AUC_PR": average_precision_score(y, y_score, pos_label=1),
        "FAR": fp / (fp + tn) if (fp + tn) > 0 else 0.0,
    }

    print(f"\n=== {dataset_name} ===")
    print("Confusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y, y_pred, digits=4, zero_division=0))
    print(pd.Series(metrics).drop(labels=["Dataset"]))

    return metrics


def plot_confusion_matrix(model, X, y, dataset_name: str) -> None:
    """Plot a confusion matrix."""
    ConfusionMatrixDisplay.from_estimator(model, X, y, display_labels=["Benign", "Malicious"])
    plt.title(f"Confusion Matrix — {dataset_name}")
    plt.tight_layout()
    plt.show()


def plot_roc_curve(model, X, y, dataset_name: str) -> None:
    """Plot the ROC curve."""
    y_score = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, y_score)
    auc_roc = roc_auc_score(y, y_score)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f"AUC-ROC = {auc_roc:.4f}")
    plt.plot([0, 1], [0, 1], "--", label="Random Guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {dataset_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_precision_recall_curve(model, X, y, dataset_name: str) -> None:
    """Plot the Precision-Recall curve."""
    y_score = model.predict_proba(X)[:, 1]
    precision, recall, _ = precision_recall_curve(y, y_score, pos_label=1)
    auc_pr = average_precision_score(y, y_score, pos_label=1)
    baseline = np.sum(y) / len(y)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f"AUC-PR = {auc_pr:.4f}")
    plt.plot([0, 1], [baseline, baseline], "--", label="Baseline")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve — {dataset_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_top_features(model, feature_names, top_n: int = 10) -> pd.DataFrame:
    """Plot the most important features according to XGBoost."""
    feature_importance = pd.DataFrame({
        "Feature": feature_names,
        "Importance": model.feature_importances_,
    }).sort_values(by="Importance", ascending=False)

    top_features = feature_importance.head(top_n).iloc[::-1]
    plt.figure(figsize=(10, 6))
    plt.barh(top_features["Feature"], top_features["Importance"])
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title(f"Top {top_n} Features")
    plt.tight_layout()
    plt.show()

    return feature_importance

## 4. Load Datasets

In [ ]:
df_unsw15_raw = load_dataset(NB15_PATH, "GenIDS-NB15")
df_cic18_raw = load_dataset(CIC18_PATH, "GenIDS-CIC18")

## 5. Preprocessing

Non-predictive identifiers and dataset-specific metadata columns are removed. Categorical attributes are encoded and numeric types are standardized.

In [ ]:
common_columns_to_remove = [
    "multiclass",
    "date",
    "hours",
    "expiration_id",
    "src_ip",
    "src_mac",
    "src_oui",
    "dst_ip",
    "dst_mac",
    "dst_oui",
    "ip_version",
    "vlan_id",
    "tunnel_id",
]

unsw15_columns_to_remove = common_columns_to_remove
cic18_columns_to_remove = [
    "multiclass",
    "label",
    "Timestamp",
    "src_ip",
    "src_mac",
    "src_oui",
    "dst_ip",
    "dst_mac",
    "dst_oui",
    "ip_version",
    "vlan_id",
    "tunnel_id",
]

# CIC18 originally uses the column "label" in this notebook version.
df_cic18_raw = normalize_label_column(df_cic18_raw, LABEL_COLUMN)

df_unsw15 = remove_non_model_columns(df_unsw15_raw, unsw15_columns_to_remove)
df_cic18 = remove_non_model_columns(df_cic18_raw, cic18_columns_to_remove)

categorical_columns = ["application_name", "application_category_name", LABEL_COLUMN]
df_unsw15, df_cic18 = encode_categorical_columns(
    [df_unsw15, df_cic18],
    categorical_columns,
)

df_unsw15 = standardize_numeric_types(df_unsw15, LABEL_COLUMN)
df_cic18 = standardize_numeric_types(df_cic18, LABEL_COLUMN)

print_dataset_summary(df_unsw15, "GenIDS-NB15")
print_dataset_summary(df_cic18, "GenIDS-CIC18")

## 6. Benign Flow Integration

Benign flows from CIC18 are integrated into UNSW15. The selected CIC18 flows are removed from the CIC18 evaluation set to avoid data leakage.

In [ ]:
integrated_unsw15, cic18_test, integrated_cic18_benign, removed_unsw15_benign = integrate_benign_flows(
    target_dataframe=df_unsw15,
    source_dataframe=df_cic18,
    integration_rate=INTEGRATION_RATE,
    target_name="UNSW15",
    source_name="CIC18",
)

print_dataset_summary(integrated_unsw15, "Integrated GenIDS-NB15")
print_dataset_summary(cic18_test, "GenIDS-CIC18 after leakage-safe removal")

print(f"\nIntegrated CIC18 benign flows: {integrated_cic18_benign.shape}")
print(f"Removed UNSW15 benign flows: {removed_unsw15_benign.shape}")

## 7. Optional Visualization of Integrated Flows

In [ ]:
if TIME_COLUMN in integrated_unsw15.columns and "bidirectional_bytes" in integrated_unsw15.columns:
    plt.figure(figsize=(16, 8))

    benign_unsw = integrated_unsw15[(integrated_unsw15[LABEL_COLUMN] == 0) & (integrated_unsw15[SOURCE_COLUMN] == "unsw15")]
    malicious_unsw = integrated_unsw15[(integrated_unsw15[LABEL_COLUMN] == 1) & (integrated_unsw15[SOURCE_COLUMN] == "unsw15")]
    benign_cic18 = integrated_unsw15[integrated_unsw15[SOURCE_COLUMN] == f"cic18_{int(INTEGRATION_RATE * 100)}_benign"]

    plt.scatter(benign_unsw[TIME_COLUMN], benign_unsw["bidirectional_bytes"], label="Benign (UNSW15)", alpha=0.6)
    plt.scatter(malicious_unsw[TIME_COLUMN], malicious_unsw["bidirectional_bytes"], label="Malicious (UNSW15)", marker="x", alpha=0.6)
    plt.scatter(benign_cic18[TIME_COLUMN], benign_cic18["bidirectional_bytes"], label="Integrated Benign (CIC18)", alpha=0.8)

    plt.title("Flow Volume over Time after Benign Flow Integration")
    plt.xlabel("Timestamp (bidirectional_first_seen_ms)")
    plt.ylabel("Data Volume (bidirectional_bytes)")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print("Required columns for visualization are not available.")

## 8. Train-Test Split and Normalization

The model is trained using the integrated UNSW15 dataset. CIC18 is used as external cross-dataset generalization test.

In [ ]:
external_test_sets = {
    "CIC18": cic18_test,
}

X_train_scaled, X_test_scaled, y_train, y_test, scaled_external_sets, feature_names = split_and_scale(
    train_dataframe=integrated_unsw15,
    external_test_dataframes=external_test_sets,
)

print(f"Training set: {X_train_scaled.shape}")
print(f"Internal test set: {X_test_scaled.shape}")
for dataset_name, (X_external, y_external) in scaled_external_sets.items():
    print(f"External test set ({dataset_name}): {X_external.shape}")

## 9. Model Training

In [ ]:
model = build_xgboost_model(RANDOM_STATE)
model.fit(X_train_scaled, y_train)
print("Model training completed.")

## 10. Feature Importance

In [ ]:
feature_importance_df = plot_top_features(model, feature_names, top_n=10)
feature_importance_df.head(20)

## 11. Intraset Evaluation — Integrated UNSW15

In [ ]:
metrics = []
metrics.append(evaluate_binary_classifier(model, X_test_scaled, y_test, "Integrated UNSW15"))
plot_confusion_matrix(model, X_test_scaled, y_test, "Integrated UNSW15")
plot_roc_curve(model, X_test_scaled, y_test, "Integrated UNSW15")
plot_precision_recall_curve(model, X_test_scaled, y_test, "Integrated UNSW15")

## 12. Interset Evaluation — CIC18

In [ ]:
X_cic18_scaled, y_cic18 = scaled_external_sets["CIC18"]
metrics.append(evaluate_binary_classifier(model, X_cic18_scaled, y_cic18, "CIC18"))
plot_confusion_matrix(model, X_cic18_scaled, y_cic18, "CIC18")
plot_roc_curve(model, X_cic18_scaled, y_cic18, "CIC18")
plot_precision_recall_curve(model, X_cic18_scaled, y_cic18, "CIC18")

## 13. Consolidated Results

In [ ]:
metrics_df = pd.DataFrame(metrics)
metrics_df

In [ ]:
output_path = RESULTS_DIR / f"benign_flow_integration_{int(INTEGRATION_RATE * 100)}pct_metrics.csv"
metrics_df.to_csv(output_path, index=False)
print(f"Metrics saved to: {output_path}")

## 14. Notes for Other Integration Rates

To run the same experiment with other integration rates, update:

```python
INTEGRATION_RATE = 0.40  # or 0.60, 0.80
```

Then re-run the notebook from the beginning. This preserves the same leakage-safe integration logic used in the 20% scenario.